## Embeddings

នៅក្នុងឧទាហរណ៍មុនរបស់​យើង យើងបាន​ប្រតិបត្តិ​លើ​វ៉ិចទ័រ bag-of-words ដែលមានខ្សែបណ្តោយខ្ពស់ `vocab_size` និង​យើងបាន​បម្លែង​យ៉ាងច្បាស់ពី​វ៉ិចទ័រ​តំណរភាគទូលាយ ទាប ទៅជាតំណាង​តែមួយ-ក្តៅ​ដែលរាយបញ្ចាំងតិចតួច។ តំណាងតែមួយ-ក្តៅ​នេះ​មិន​មានប្រសិទ្ធភាព​ក្នុង​ការផ្ទុកចងចាំទេ បន្ថែម​ពីនេះ រាល់ពាក្យ​ត្រូវបានចាត់ទុក​ឯករាជ្យពីគ្នា ប្រែថា​វ៉ិចទ័រដែលបាន encode តែមួយ-ក្តៅ​មិនបង្ហាញន័យស្រដៀងគ្នាណាមួយ​រវាងពាក្យទេ។

នៅក្នុងឯកតានេះ យើងនឹងបន្ត​សហការជាមួយ​ទិន្នន័យ **News AG**។ ដើម្បីចាប់ផ្តើម មកយើងផ្ទុកទិន្នន័យ និងទទួលយក​និយមន័យខ្លះ​ពីសៀវភៅកំណត់ហេតុមុន។


In [1]:
import torch
import torchtext
import numpy as np
from torchnlp import *
train_dataset, test_dataset, classes, vocab = load_dataset()
vocab_size = len(vocab)
print("Vocab size = ",vocab_size)

Loading dataset...


d:\WORK\ai-for-beginners\5-NLP\14-Embeddings\data\train.csv: 29.5MB [00:01, 18.8MB/s]                            
d:\WORK\ai-for-beginners\5-NLP\14-Embeddings\data\test.csv: 1.86MB [00:00, 11.2MB/s]                          


Building vocab...
Vocab size =  95812


## តើ embedding ជាអ្វី?

គំនិតនៃ **embedding** គឺជារៀបរាប់​ពាក្យ​ជាវ៉ិចទ័រដ៏រឹងមាំដែលមានវិមាត្រកថ្លៃទាប ជាដើមដែលបញ្ជាក់អត្ថន័យ​យោងទៅលើពាក្យមួយ។ យើង​នឹងពិភាក្សាពីរបៀបកសាង word embeddings មានន័យ​ក្នុង​ពេលក្រោយ ប៉ុន្តែ​នាពេលនេះ hãyគិតថា embeddings ជាវិធីកាត់បន្ថយវិមាត្រនៃវ៉ិចទ័រពាក្យមួយ។

ដូច្នេះ, ស្រទាប់ embedding នឹងទទួលពាក្យមួយជាឧបត្ថម្ភ និងបង្កើតវ៉ិចទ័រចេញ​មួយដែលមាន `embedding_size` បានកំណត់។ ក្នុងន័យមួយវាស្រដៀងទៅនឹងស្រទាប់ `Linear` តែជំនួសវិចទ័រពោលជា one-hot encoded វា​អាចទទួលលេខពាក្យដោយផ្ទាល់ជាឧបត្ថម្ភបាន។

ដោយប្រើស្រទាប់ embedding ជាស្រទាប់ដំបូងក្នុងបណ្តាញរបស់យើង យើងអាចបម្លែងពីគំរូ bag-of-words ទៅគំរូ **embedding bag** ដែលនៅទីនោះយើងបម្លែងពាក្យនីមួយៗក្នុងអត្ថបទជាវ៉ិចទ័រពាក់ព័ន្ធ embedding ហើយបន្ទាប់មកគណនាអនុគមន៍សរុបលើវ៉ិចទ័រទាំងនោះ ដូចជា `sum`, `average` ឬ `max`។

![រូបភាពបង្ហាញឧទាហរណ៍ embedding classifier សម្រាប់ពាក្យជាច្រើនក្នុងលំដាប់។](../../../../../translated_images/km/embedding-classifier-example.b77f021a7ee67eee.webp)

បណ្តាញនឺរ៉ាល់នៃ classifer របស់យើង នឹងចាប់ផ្តើមជាមួយស្រទាប់ embedding បន្ទាប់ស្រទាប់ aggregation និងចុងក្រោយជាស្រទាប់ linear classifier៖


In [2]:
class EmbedClassifier(torch.nn.Module):
    def __init__(self, vocab_size, embed_dim, num_class):
        super().__init__()
        self.embedding = torch.nn.Embedding(vocab_size, embed_dim)
        self.fc = torch.nn.Linear(embed_dim, num_class)

    def forward(self, x):
        x = self.embedding(x)
        x = torch.mean(x,dim=1)
        return self.fc(x)

### ការដោះស្រាយទំហំលំដាប់អថេរ

ជាលទ្ធផលនៃស្ថាបត្យកម្មនេះ minibatches ទៅបណ្ដាញរបស់យើងនឹងត្រូវបានបង្កើតឡើងដោយវិធីជាក់លាក់មួយ។ នៅឯកតាមុននេះ នៅពេលប្រើ bag-of-words តង់ស័រទាំងអស់ក្នុង minibatch មានទំហំពេលស្មើគ្នា `vocab_size` មិនគិតទំហំជាក់លាក់នៃលំដាប់អត្ថបទរបស់យើងឡើយ។ ពេលយើងផ្លាស់ទៅប្រើបន្ទាត់ពាក្យ word embeddings យើងនឹងបានចំនួនពាក្យប្រែប្រួលនៅក្នុងគំរូអត្ថបទនីមួយៗ ហើយពេលបញ្ចូលគំរូទាំងនោះចូលក្នុង minibatches យើងនឹងត្រូវអនុវត្តការបញ្ចូលបន្ថែម padding មួយចំនួន។

នេះអាចធ្វើបានដោយប្រើបច្ចេកទេសដដែលគឺផ្តល់មុខងារ `collate_fn` ទៅដល់ datasource៖


In [3]:
def padify(b):
    # b is the list of tuples of length batch_size
    #   - first element of a tuple = label, 
    #   - second = feature (text sequence)
    # build vectorized sequence
    v = [encode(x[1]) for x in b]
    # first, compute max length of a sequence in this minibatch
    l = max(map(len,v))
    return ( # tuple of two tensors - labels and features
        torch.LongTensor([t[0]-1 for t in b]),
        torch.stack([torch.nn.functional.pad(torch.tensor(t),(0,l-len(t)),mode='constant',value=0) for t in v])
    )

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, collate_fn=padify, shuffle=True)

### ការបណ្តុះបណ្តាលម៉ាស៊ីនចាត់ថ្នាក់ embedding

ឥឡូវនេះដែលយើងបានកំណត់ dataloader ត្រឹមត្រូវហើយ យើងអាចបណ្តុះបណ្តាលម៉ូដែលដោយប្រើមុខងារ training ដែលយើងបានកំណត់នៅឯកតាមុន៖


In [4]:
net = EmbedClassifier(vocab_size,32,len(classes)).to(device)
train_epoch(net,train_loader, lr=1, epoch_size=25000)

3200: acc=0.6415625
6400: acc=0.6865625
9600: acc=0.7103125
12800: acc=0.726953125
16000: acc=0.739375
19200: acc=0.75046875
22400: acc=0.7572321428571429


(0.889799795315499, 0.7623160588611644)

> **កំណត់សម្គាល់**: យើងកំពុង ONLY បណ្តុះបណ្តាលសម្រាប់កំណត់ត្រា 25,000 នៅទីនេះ (តិចជាងមួយស្ទង់ពេញ) ដើម្បីជួយការសន្សំសំចៃពេលវេលា ប៉ុន្តែអ្នកអាចបន្តការបណ្តុះបណ្តាល សរសេរពិធីសាស្រ្តមួយសម្រាប់បណ្តុះបណ្តាលជាច្រើនស្ទង់ និងសាកល្បងជាមួយប៉ារ៉ាម៉ែត្រការសិក្សា ដើម្បីទទួលបានភាពត្រឹមត្រូវខ្ពស់ជាងមុន។ អ្នកគួរតែអាចទៅដល់ភាពត្រឹមត្រូវប្រហែល ៩០%។


### EmbeddingBag ស្រទាប់ និង តំណាងរាងលំដាប់រៀងពីរយៈវែរីយ៉ាប៊ុល

នៅក្នុងស្ថาปត្យកម្មមុន យើង​ត្រូវការបន្ថែម​ចន្លោះ​ទទេ​ទៅលើ​លំដាប់​ទាំងអស់ដើម្បីឲ្យវា​មានប្រវែង​ដូចគ្នា​ដើម្បីដាក់វាទៅក្នុង minibatch។ វា​មិនមែន​ជា​របៀបមានប្រសិទ្ធភាព​បំផុត​ក្នុងការតំណាង​លំដាប់​វែរីយ៉ាប៊ុល​នោះទេ - វិធានមួយផ្សេងគឺប្រើ​វ៉ិចទ័រ **offset** ដែលនឹងរក្សាទុកចន្លោះ offset របស់​លំដាប់ទាំងអស់​ដែលបានរក្សាទុក​ក្នុង​វ៉ិចទ័រធំបន្ទាត់មួយ។

![Image showing an offset sequence representation](../../../../../translated_images/km/offset-sequence-representation.eb73fcefb29b46ee.webp)

> **ចំណាំ**: នៅក្នុងរូបភាពខាងលើ យើងបង្ហាញ​លំដាប់​តួអក្សរ ប៉ុន្តែនៅក្នុងឧទាហរណ៍របស់យើង កំពុង​ដំណើរការជាលំដាប់​ពាក្យ។ ទោះយ៉ាងណា 원칙​ទូទៅ​ក្នុងការតំណាងលំដាប់ជាមួយវ៉ិចទ័រ offset នៅតែដូចគ្នា។

ដើម្បីធ្វើការ​ជាមួយតំណាង offset យើងប្រើស្រទាប់ [`EmbeddingBag`](https://pytorch.org/docs/stable/generated/torch.nn.EmbeddingBag.html)។ វាស្រដៀងនឹង `Embedding` ប៉ុន្តែវាបានទទួល​ខ្សែវ៉ិចទ័រ និង offset វ៉ិចទ័រជាជាច្រើននូវការបញ្ចូល ហើយវាក៏រួមបញ្ចូលជាមួយស្រទាប់ averaging ដែលអាចជាផ្នែក `mean`, `sum` ឬ `max` ផងដែរ។

នេះគឺជាបណ្តាញដែលបានកែសម្រួលដែលប្រើ `EmbeddingBag`៖


In [5]:
class EmbedClassifier(torch.nn.Module):
    def __init__(self, vocab_size, embed_dim, num_class):
        super().__init__()
        self.embedding = torch.nn.EmbeddingBag(vocab_size, embed_dim)
        self.fc = torch.nn.Linear(embed_dim, num_class)

    def forward(self, text, off):
        x = self.embedding(text, off)
        return self.fc(x)

ដើម្បីរៀបចំទិន្នន័យសម្រាប់ហ្វឹកហាត់ យើងត្រូវការផ្តល់មុខងារបម្លែងមួយ ដែលនឹងរៀបចំពាក្យកំណត់អូស៖


In [6]:
def offsetify(b):
    # first, compute data tensor from all sequences
    x = [torch.tensor(encode(t[1])) for t in b]
    # now, compute the offsets by accumulating the tensor of sequence lengths
    o = [0] + [len(t) for t in x]
    o = torch.tensor(o[:-1]).cumsum(dim=0)
    return ( 
        torch.LongTensor([t[0]-1 for t in b]), # labels
        torch.cat(x), # text 
        o
    )

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, collate_fn=offsetify, shuffle=True)

សូមកត់សម្គាល់ថា ខុសពីគំរូទាំងអស់មុននេះ បណ្ដាញរបស់យើងឥឡូវនេះទទួលបានប៉ារាម៉ែត្រ​ពីរចូលគ្នា៖ វិចទ័រទិន្នន័យ និងវិចទ័រជំហរ ដែលមានទំហំខុសគ្នា។ ដូចគ្នានេះ អ្នកដំណើរការទិន្នន័យរបស់យើងក៏ផ្ដល់ឲ្យយើងនូវតម្លៃចំនួន៣មិនមែនត្រឹមតែ២ទេ៖ វិចទ័រទាំងអត្ថបទ និងជំហរត្រូវបានផ្ដល់ឲ្យជាលក្ខណៈពិសេស។ ដូច្នេះ យើងត្រូវកែប្រែ slightly ប្រតិបត្ដិការបណ្តុះបណ្តាលរបស់យើង ដើម្បីដោះស្រាយបញ្ហានេះ៖


In [7]:
net = EmbedClassifier(vocab_size,32,len(classes)).to(device)

def train_epoch_emb(net,dataloader,lr=0.01,optimizer=None,loss_fn = torch.nn.CrossEntropyLoss(),epoch_size=None, report_freq=200):
    optimizer = optimizer or torch.optim.Adam(net.parameters(),lr=lr)
    loss_fn = loss_fn.to(device)
    net.train()
    total_loss,acc,count,i = 0,0,0,0
    for labels,text,off in dataloader:
        optimizer.zero_grad()
        labels,text,off = labels.to(device), text.to(device), off.to(device)
        out = net(text, off)
        loss = loss_fn(out,labels) #cross_entropy(out,labels)
        loss.backward()
        optimizer.step()
        total_loss+=loss
        _,predicted = torch.max(out,1)
        acc+=(predicted==labels).sum()
        count+=len(labels)
        i+=1
        if i%report_freq==0:
            print(f"{count}: acc={acc.item()/count}")
        if epoch_size and count>epoch_size:
            break
    return total_loss.item()/count, acc.item()/count


train_epoch_emb(net,train_loader, lr=4, epoch_size=25000)

3200: acc=0.6153125
6400: acc=0.6615625
9600: acc=0.6932291666666667
12800: acc=0.715078125
16000: acc=0.7270625
19200: acc=0.7382291666666667
22400: acc=0.7486160714285715


(22.771553103007037, 0.7551983365323096)

## Semantic Embeddings: Word2Vec

នៅក្នុងឧទាហរណ៍មុនរបស់យើង ស្រទាប់ embedding ម៉ូដែលបានរៀនផែនទីពាក្យទៅរាងតំណាងវ៉ិចទ័រ ទោះជាយ៉ាងណា រាងតំណាងនេះមិនមានន័យsemantic ច្រើនទេ។ វានឹងល្អបើរៀនរាងតំណាងវ៉ិចទ័រដូច្នេះ ដែលពាក្យស្រដៀងគ្នា ឬពាក្យប្រហាក់ប្រហែលនឹងផ្គូផ្គងទៅនឹងវ៉ិចទ័រដែលនៅជិតគ្នាប្រកបដោយចម្ងាយវ៉ិចទ័រ (ឧ. ចម្ងាយអ៊ីយូគ្ល៊ីដៀន) ។

ដើម្បីធ្វើបានដូច្នេះ យើងត្រូវបណ្ដុះម៉ូដែល embedding របស់យើងលើការប្រមូលផ្តុំអត្ថបទធំៗដោយវិធីពិសេសមួយ។ វិធីដំបូងៗក្នុងការបណ្ដុះ semantic embeddings មួយឈ្មោះ [Word2Vec](https://en.wikipedia.org/wiki/Word2vec)។ វាមានព្រណិតប្រាជ្ញាផ្លូវការចម្បងពីរដែលប្រើប្រព័ន្ធផ្សព្វផ្សាយរាងតំណាងរាងពាក្យមួយ៖

 - **Continuous bag-of-words** (CBoW) — ក្នុងសំណុំប្រាជ្ញាផ្លូវការនេះ យើងបណ្ដុះម៉ូដែលឲ្យទស្សនាពាក្យមួយពីបរិបទជុំវិញ។ ផ្តល់អោយ ngram $(W_{-2},W_{-1},W_0,W_1,W_2)$ គោលបំណងម៉ូដែលគឺទស្សនាពាក្យ $W_0$ ពី $(W_{-2},W_{-1},W_1,W_2)$។
 - **Continuous skip-gram** គឺលើកលែងពី CBoW។ ម៉ូដែលប្រើបណ្តោយបរិបទវ៉ីនដូជុំវិញ ដើម្បីទស្សនាពាក្យបច្ចុប្បន្ន។

CBoW ឆាប់រហ័ស ខណៈដែល skip-gram យឺតជាង ប៉ុន្តែប្រសើរជាងក្នុងការតំណាងពាក្យដែលមិនមានប្រជាជនច្រើន។

![រូបភាពបង្ហាញអាល់ហ្គរីធึម CBoW និង Skip-Gram ដើម្បីបំលែងពាក្យទៅវ៉ិចទ័រ។](../../../../../translated_images/km/example-algorithms-for-converting-words-to-vectors.fbe9207a726922f6.webp)

ដើម្បីសាកល្បង embedding word2vec ដែលបានបណ្ដុះរួចនៅលើឃ្លាំងទិន្នន័យ Google News អស់យើងអាចប្រើបណ្ណាល័យ **gensim** ។ ខាងក្រោម យើងស្វែងរកពាក្យដែលស្រដៀងគ្នាច្រើនជាងគេជាមួយពាក្យ 'neural'

> **Note:** រស់រាន់ពេលអ្នកបង្កើតវ៉ិចទ័រពាក្យដំបូង ការទាញយកវានឹងចំណាយពេលខ្លះ!


In [8]:
import gensim.downloader as api
w2v = api.load('word2vec-google-news-300')

In [9]:
for w,p in w2v.most_similar('neural'):
    print(f"{w} -> {p}")

neuronal -> 0.7804799675941467
neurons -> 0.7326500415802002
neural_circuits -> 0.7252851724624634
neuron -> 0.7174385190010071
cortical -> 0.6941086649894714
brain_circuitry -> 0.6923246383666992
synaptic -> 0.6699118614196777
neural_circuitry -> 0.6638563275337219
neurochemical -> 0.6555314064025879
neuronal_activity -> 0.6531826257705688


យើងអាចគណនាការតំណាងវ៉ែកទ័រពីពាក្យដើម្បីប្រើក្នុងការបណ្តុះមូឌែលចាត់ថ្នាក់ (យើងបង្ហាញតែ ២០ ធាតុដំបូងនៃវ៉ែកទ័រដើម្បីភាពច្បាស់):


In [10]:
w2v.word_vec('play')[:20]

array([ 0.01226807,  0.06225586,  0.10693359,  0.05810547,  0.23828125,
        0.03686523,  0.05151367, -0.20703125,  0.01989746,  0.10058594,
       -0.03759766, -0.1015625 , -0.15820312, -0.08105469, -0.0390625 ,
       -0.05053711,  0.16015625,  0.2578125 ,  0.10058594, -0.25976562],
      dtype=float32)

រឿងល្អណាស់អំពីសេចក្តីចងក្រងសម្ភារៈន័យគឺអ្នកអាចត្រួតពិនិត្យកូដចុះបញ្ជីវ៉ិចទ័រដើម្បីផ្លាស់ប្តូរអត្ថន័យ។ ឧទាហរណ៍ យើងអាចស្នើសុំស្វែងរកពាក្យមួយដែលការតំនាងវ៉ិចទ័របស់វានឹងនៅជិតពាក្យ *king* និង *woman* ចំបង ហើយឆ្ងាយពីពាក្យ *man*៖


In [10]:
w2v.most_similar(positive=['king','woman'],negative=['man'])[0]

('queen', 0.7118192911148071)

ទាំង CBoW និង Skip-Grams ជា embedding ទាំងពីរដែល “ទាយ” គឺដោយពួកវាទទួលយកតែបរិបទក្នុងតំបន់ជិតៗ។ Word2Vec មិនប្រើប្រាស់អត្ថប្រយោជន៍ពីបរិបទសកលឡើយ។

**FastText** សាងសង់លើ Word2Vec ដោយរៀនតំណាងវ៉ិកទ័រសម្រាប់ពាក្យនីមួយៗ និង n-gram តួអក្សរដែលមានក្នុងពាក្យនីមួយៗ។ តម្លៃនៃតំណាងទាំងនេះនឹងត្រូវបានរាប់បូកជាមធ្យមទៅវ៉ិកទ័រតែមួយនៅក្នុងជំហានបណ្តុះបណ្តាលមួយៗ។ បើទោះបីជាវានាំឲ្យមានការគណនាថែមទៀតច្រើនក្នុងការបណ្តុះបណ្តាលមុនក៏ដោយ វាអនុញ្ញាតឲ្យ embedding ពាក្យអាចបង្ហាញព័ត៌មានក្រោមពាក្យ។

វិធីផ្សេងទៀត គឺ **GloVe** អនុវត្តគំនិតម៉ាត្រីសco-occurrence ហើយប្រើវិធីណឺរ៉ាល់ដើម្បីបំបែកម៉ាត្រីស co-occurrence ទៅជាវ៉ិកទ័រពាក្យដែលមានការបង្ហាញពត៌មានច្រើន និងមិនមែនជារូបមន្តលាញ់។

អ្នកអាចលេងជាមួយឧទាហរណ៍ដោយផ្លាស់ប្តូរ embeddings ទៅជា FastText និង GloVe ព្រោះ gensim គាំទ្រម៉ូដែល embedding ពាក្យជាច្រើនផ្សេងទៀត។


## ការប្រើប្រាស់ការកំណត់តម្លៃជាមុនក្នុង PyTorch

យើងអាចកែប្រែឧទាហរណ៍ខាងលើដើម្បីបញ្ចូល matrix នៅលើស្រទាប់ embedding របស់យើងជាមុនជាមួយនឹងការកំណត់តម្លៃពាក្យដែលមានអត្ថន័យ ដូចជា Word2Vec។ យើងត្រូវគិតគូរថា និយមន័យនៃការកំណត់តម្លៃជាមុន និងអត្ថបទរបស់យើងប្រហែលជាមិនត្រូវគ្នា ដូច្នេះយើងនឹងផ្តាច់តម្លៃទម្លាក់សម្រាប់ពាក្យដែលខ្វះដោយតម្លៃចៃដន្យ:


In [11]:
embed_size = len(w2v.get_vector('hello'))
print(f'Embedding size: {embed_size}')

net = EmbedClassifier(vocab_size,embed_size,len(classes))

print('Populating matrix, this will take some time...',end='')
found, not_found = 0,0
for i,w in enumerate(vocab.get_itos()):
    try:
        net.embedding.weight[i].data = torch.tensor(w2v.get_vector(w))
        found+=1
    except:
        net.embedding.weight[i].data = torch.normal(0.0,1.0,(embed_size,))
        not_found+=1

print(f"Done, found {found} words, {not_found} words missing")
net = net.to(device)

Embedding size: 300
Populating matrix, this will take some time...Done, found 41080 words, 54732 words missing


ឥឡូវនេះយើងចាប់ផ្តើមបណ្តុះម៉ូឌែលរបស់យើង។ សូមចំណាំថា ពេលវេលាដែលត្រូវការដើម្បីបណ្ដុះម៉ូឌែលនេះធ្វើបានយូរជាងឧទាហរណ៍មុនសម្រាប់ស្វ័យប្រវត្តន៍ធំជាងនេះហើយដែលមានប៉ារ៉ាម៉ែត្រច្រើនជាង។ ផងដែរ ដោយសារតែហេតុនេះ យើងប្រហែលជាត្រូវការបណ្តុះម៉ូឌែលរបស់យើងលើឧទាហរណ៍ច្រើនជាងនេះបើចង់ជៀសវាងការស្វ័យប្រវត្តិ។


In [12]:
train_epoch_emb(net,train_loader, lr=4, epoch_size=25000)

3200: acc=0.6359375
6400: acc=0.68109375
9600: acc=0.7067708333333333
12800: acc=0.723671875
16000: acc=0.73625
19200: acc=0.7463541666666667
22400: acc=0.7560714285714286


(214.1013875559821, 0.7626759436980166)

នៅក្នុងករណីរបស់យើង យើងមិនឃើញការកើនឡើងយ៉ាងខ្លាំងនៅក្នុងភាពត្រឹមត្រូវទេ ដែលអាចជាសារជាផ្ទុយគ្នានៃវចនានុក្រម។  
ដើម្បីឆ្លើយតបទៅនឹងបញ្ហានៃវចនានុក្រមខុសគ្នា យើងអាចប្រើចំណុចដោះស្រាយដូចខាងក្រោមមួយ៖  
* ធ្វើការបណ្តុះបណ្តាលម៉ូដែល word2vec ម្តងទៀតលើវចនានុក្រមរបស់យើង  
* ផ្ទុកទិន្នន័យរបស់យើងជាមួយវចនានុក្រមពីម៉ូដែល word2vec ដែលបានបណ្តុះបណ្តាលរួចហើយ។ វចនានុក្រមដែលប្រើសម្រាប់ផ្ទុកទិន្នន័យអាចកំណត់បាននៅពេលផ្ទុក។  

វិធីសាស្រ្តចុងក្រោយមើលទៅងាយស្រួលជាងលើសពីព្រោះស៊ុម PyTorch `torchtext` មានការគាំទ្របង្កើត embeddings ជាផ្នែកមួយក្នុងខ្លួន។ យើងអាច ឧទាហរណ៍ បង្កើតវចនានុក្រមមានមូលដ្ឋានលើ GloVe ដូចខាងក្រោមនេះ៖


In [14]:
vocab = torchtext.vocab.GloVe(name='6B', dim=50)

100%|█████████▉| 399999/400000 [00:15<00:00, 25411.14it/s]


Loaded vocabulary has the following basic operations:
* `vocab.stoi` ឌិចសិនាឡរី អនុញ្ញាតឱ្យយើងបម្លែងពាក្យទៅជាអុិនដិចក្នុងឌិចសិនាឡរី
* `vocab.itos` ធ្វើការប្រែប្រួលផ្ទុយ - បម្លែងលេខទៅជាពាក្យ
* `vocab.vectors` គឺជាអារេនៃវ៉ិកទ័រអ៊ាំប៊ីឌិញ ដូច្នេះដើម្បីទទួលបានអ៊ាំប៊ីឌីងរបស់ពាក្យ `s` យើងត្រូវប្រើ `vocab.vectors[vocab.stoi[s]]`

នេះគឺជាឧទាហរណ៍នៃការគ្រប់គ្រងអ៊ាំប៊ីឌីងដើម្បីបង្ហាញលើសមីការនេះ **kind-man+woman = queen** (ខ្ញុំបានកែប្រែសមាមាត្រតិចតួចដើម្បីឲ្យវាដំណើរការ):


In [15]:
# get the vector corresponding to kind-man+woman
qvec = vocab.vectors[vocab.stoi['king']]-vocab.vectors[vocab.stoi['man']]+1.3*vocab.vectors[vocab.stoi['woman']]
# find the index of the closest embedding vector 
d = torch.sum((vocab.vectors-qvec)**2,dim=1)
min_idx = torch.argmin(d)
# find the corresponding word
vocab.itos[min_idx]

'queen'

ដើម្បីបណ្តុះបណ្តាលម៉ាស៊ីនចម្រាញ់ប្រភេទដោយប្រើ embeddings ទាំងនោះ សិនមុនយើងត្រូវបំលែងឱ្យប្រើពាក្យសម្គាល់ GloVe លើសំណុំទិន្នន័យរបស់យើង៖


In [16]:
def offsetify(b):
    # first, compute data tensor from all sequences
    x = [torch.tensor(encode(t[1],voc=vocab)) for t in b] # pass the instance of vocab to encode function!
    # now, compute the offsets by accumulating the tensor of sequence lengths
    o = [0] + [len(t) for t in x]
    o = torch.tensor(o[:-1]).cumsum(dim=0)
    return ( 
        torch.LongTensor([t[0]-1 for t in b]), # labels
        torch.cat(x), # text 
        o
    )

ដូចដែលយើងបានឃើញខាងលើ វិកទ័រសម្រាប់បញ្ចូលទាំងអស់ត្រូវបានផ្ទុកនៅក្នុងម៉ាទ្រីស `vocab.vectors`។ វាធ្វើឲ្យមានភាពងាយស្រួលយ៉ាងខ្លាំងក្នុងការបញ្ចូលទំងន់ទាំងនោះទៅក្នុងទម្រង់ទំងន់នៃស្រទាប់ embedding ដោយប្រើការចម្លងធម្មតា៖


In [17]:
net = EmbedClassifier(len(vocab),len(vocab.vectors[0]),len(classes))
net.embedding.weight.data = vocab.vectors
net = net.to(device)

ឥឡូវនេះរួចដំណើរការបណ្ដុះបណ្ដាលម៉ូដែលរបស់យើង ហើយមើលថាតើយើងទទួលបានលទ្ធផលកាន់តែល្អឬយ៉ាងដូចម្តេច៖


In [18]:
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, collate_fn=offsetify, shuffle=True)
train_epoch_emb(net,train_loader, lr=4, epoch_size=25000)

3200: acc=0.6271875
6400: acc=0.68078125
9600: acc=0.7030208333333333
12800: acc=0.71984375
16000: acc=0.7346875
19200: acc=0.7455729166666667
22400: acc=0.7529464285714286


(35.53972978646833, 0.7575175943698017)

មួយក្នុងចំណោមហេតុផលដែលយើងមិនឃើញការកើនឡើងយ៉ាងសំខាន់នៃភាពទាន់សម័យគឺដោយសារតែកម្រិតពាក្យខ្លះពីទិន្នន័យរបស់យើងបានបាត់បង់ក្នុងវាក្យសព្ទ GloVe ដែលបានបណ្តុះមុន ហើយដូច្នេះពួកវាត្រូវបានរំលងផ្តាច់។ ដើម្បីទ្រាក់ទ្រង់លើការពិតនេះ យើងអាចបណ្តុះ embeddings ផ្ទាល់ខ្លួនលើទិន្នន័យរបស់យើងបាន។


## Contextual Embeddings

ការពេញលេញចម្បងមួយនៃតំណាងការជួសជុលពីមុនដែលមានសមត្ថភាពនៃ pretrained embedding representations ដូចជា Word2Vec គឺបញ្ហានៃការប្រែប្រួលអត្ថន័យពាក្យ។ នៅពេល pretrained embeddings អាចចាប់យកអត្ថន័យខ្លះៗនៃពាក្យនៅក្នុងបរិបទជាមួយ ក៏ប៉ុន្តែអត្ថន័យដែលអាចមានទាំងអស់របស់ពាក្យត្រូវបានអចិន្រ្ត័យទៅក្នុង embedding ដូចគ្នា។ វាអាចបង្កើតបញ្ហាក្នុងម៉ូដែលក្រោយ, ពោលគឺពាក្យជាច្រើនដូចជា ពាក្យ 'play' មានអត្ថន័យខុសគ្នាច្រើនអាស្រ័យលើបរិបទដែលពួកវាត្រូវបានប្រើ។

ឧទាហរណ៍ ពាក្យ 'play' នៅក្នុងប្រយោគទាំងពីរនោះមានអត្ថន័យខុសគ្នាក្រោមមួយ:
- ខ្ញុំបានទៅបង្ហាញ **play** នៅលើឆាកតេអាទឺរ។
- John ចង់ **play** ជាមួយមិត្តភក្តិរបស់គាត់។

pretrained embeddings នៅលើនេះតំណាងឲ្យទាំងពីរអត្ថន័យនៃពាក្យ 'play' នៅក្នុង embedding ដូចគ្នា។ ដើម្បីដោះស្រាយកំណត់នេះ, យើងត្រូវតែបង្កើត embeddings បញ្ជាក់លើជារបៀប **language model**, ដែលបានបណ្តុះបណ្តាលលើនិទស្សន៍ធំមួយនៃអត្ថបទ ហើយ*ដឹង*ថាពាក្យអាចត្រូវបានដាក់ជាមួយគ្នា​របៀបខុសៗគ្នា​នៅក្នុងបរិបទផ្សេងៗ។ ការពិភាក្សាពី contextual embeddings គឺក្រៅប្រធានបទសម្រាប់ មេរៀននេះ ប៉ុន្តែយើងនឹងត្រឡប់មកមើលពួកវាទៀតនៅពេលនិយាយអំពី language models ក្នុងឯកត្តបន្ទាប់។


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ការព្រមាន**៖
ឯកសារនេះត្រូវបានបកប្រែដោយប្រើសេវាកម្មបកប្រែ AI [Co-op Translator](https://github.com/Azure/co-op-translator)។ ខណៈពេលយើងព្យាយាមរក្សាការពិតច្បាស់ សូមយល់ឲ្យបានថាការបកប្រែដោយស្វ័យប្រវត្តិសាច់បាច់អាចមានកំហុស ឬការមិនត្រឹមត្រូវនៅខ្លះ។ ឯកសារដើមដែលមានភាសាដើមគួរត្រូវបានគិតថាជាធនធានផ្លូវការជាចម្បង។ សម្រាប់ព័ត៌មានសំខាន់ ការបកប្រែដោយមនុស្សជំនាញត្រូវបានផ្តល់អាទិភាព។ យើងមិនទទួលខុសត្រូវចំពោះការយល់ច្រឡំ ឬការបំភ្លឺខុសចេញពីការប្រើប្រាស់ការបកប្រែនេះឡើយ។
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
